# Link prediction — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Link prediction ranks missing edges by evidence that two endpoints should connect
Link prediction is an edge-level task on absent pairs. The examples compute common neighbors, Adamic-Adar, preferential attachment, embedding dot scores, and a tiny ranking metric.

In [ ]:
# 1) common-neighbor score
A=np.array([[0,1,1,0],[1,0,1,1],[1,1,0,0],[0,1,0,0]])
CN=A@A
plt.figure(figsize=(4,3)); plt.imshow(CN,cmap='Blues'); plt.colorbar(); plt.title('common-neighbor scores')
assert int(CN[2,3])==1

In [ ]:
# 2) Adamic-Adar downweights popular common neighbors
A=np.array([[0,1,1,0],[1,0,1,1],[1,1,0,0],[0,1,0,0]],float); deg=A.sum(1)
aa=sum(1/math.log(deg[z]) for z in range(4) if A[2,z] and A[3,z])
plt.figure(figsize=(5,3)); plt.bar(['AA(2,3)'],[aa]); plt.title('Adamic-Adar score')
assert abs(aa-0.910239)<1e-5

In [ ]:
# 3) preferential attachment uses endpoint degrees
A=np.array([[0,1,1,0],[1,0,1,1],[1,1,0,0],[0,1,0,0]],float); deg=A.sum(1); pa=deg[2]*deg[3]
plt.figure(figsize=(5,3)); plt.bar(['deg(2)','deg(3)','PA'],[deg[2],deg[3],pa]); plt.title('preferential attachment')
assert pa==2

In [ ]:
# 4) embedding dot product scores a candidate link
Z=np.array([[1.,0.],[0.8,0.2],[0.6,0.4],[0.1,0.9]]); score=Z[2]@Z[3]
plt.figure(figsize=(4,4)); plt.scatter(Z[:,0],Z[:,1],s=250); [plt.text(*Z[i],str(i),ha='center',va='center') for i in range(4)]; plt.title(f'dot(2,3)={score:.2f}')
assert abs(score-0.42)<1e-9

In [ ]:
# 5) AUC-like pair ranking: positive scores above negatives
pos=np.array([0.91,0.42]); neg=np.array([0.20,0.10,0.30]); auc=np.mean([p>n for p in pos for n in neg])
plt.figure(figsize=(5,3)); plt.scatter(np.zeros_like(pos),pos,label='pos'); plt.scatter(np.ones_like(neg),neg,label='neg'); plt.xlim(-.5,1.5); plt.legend(); plt.title(f'pairwise AUC={auc:.2f}')
assert abs(auc-1.0)<1e-9